# SecureSpeak v2 — Daily Phishing URL Scraper (keyless)
## Grows your MFS-brand phishing dataset for Paper 1

**What this does:** pulls fresh phishing URLs from public feeds (OpenPhish +
URLhaus, both free/no-key), keeps the ones that impersonate Bangladeshi MFS
brands (bKash/Nagad/Rocket/etc.), scores them with your URL model, removes
duplicates, and APPENDS to a growing master file in your v2 folder.

**How to use:** open in Colab, Runtime -> Run all, close. Do this ONCE A DAY.
Each day the feeds have NEW phishing, so the file grows. After a few weeks
you'll have hundreds more real MFS-brand URLs -> a stronger rescue result.

**Why not 3 Gmail accounts:** the feeds are public and identical for everyone,
so multiple accounts just download the same URLs. TIME (daily runs) grows the
data, not accounts.

**Writes to:** `SecureSpeak_v2_Guardian/data/borderline_master.csv`
(never touches your old work).

## Cell 1 — Setup

In [ ]:
import os, re, json, time, math, difflib
import pandas as pd, numpy as np
from datetime import datetime
from google.colab import drive
drive.mount('/content/drive')
import joblib

V2='/content/drive/MyDrive/cse498R/SecureSpeak_v2_Guardian'
DATADIR=os.path.join(V2,'data')
os.makedirs(DATADIR, exist_ok=True)
MASTER=os.path.join(DATADIR,'borderline_master.csv')
MFS_MASTER=os.path.join(DATADIR,'mfs_brand_urls.csv')

# load URL model to score new URLs
url_model=joblib.load(os.path.join(V2,'models','url_model.joblib'))
url_scaler=joblib.load(os.path.join(V2,'models','url_scaler.joblib'))
print('Setup ready. Today:', datetime.now().strftime('%Y-%m-%d %H:%M'))

## Cell 2 — MFS brand list + URL feature engineering
(Same brands and features as your detectors, so scoring is consistent.)

In [ ]:
!pip install -q tldextract requests 2>/dev/null
import tldextract, requests

MFS_BRANDS=['bkash','bikash','bkas','bkassh','bkosh','nagad','nagd','nogod',
            'rocket','roket','dutchbangla','dbbl','upay','tap']

def is_mfs_brand(url):
    u=str(url).lower()
    return any(b in u for b in MFS_BRANDS)

HIGH_RISK_TLDS={'tk','ml','ga','cf','gq','pw','top','xyz','online','site','club','live','shop','info','biz','link','click','download','stream'}
FREE_HOST_TLDS={'tk','ml','ga','cf','gq','pw'}
FINANCIAL_KW=['bank','login','secure','verify','update','account','password','signin','bkash','nagad','rocket','paypal','confirm']
BRAND_KW=['paypal','amazon','google','facebook','apple','microsoft','bkash','nagad','rocket']

def _ent(s):
    if not s: return 0.0
    f={}; 
    for c in s: f[c]=f.get(c,0)+1
    n=len(s); return -sum((v/n)*math.log2(v/n) for v in f.values())

def engineer_url_features(url):
    url=str(url).strip().lower(); ext=tldextract.extract(url)
    domain,suffix,subdomain=ext.domain,ext.suffix,ext.subdomain
    path=re.sub(r'https?://[^/]+','',url); query=path.split('?',1)[1] if '?' in path else ''
    ip_host=url.split('/')[2] if '/' in url else url
    return [min(len(url)/500,1.0),min(url.count('.')/10,1.0),min(url.count('/')/15,1.0),
        min(len(re.findall(r'[-_@!%&=+]',url))/20,1.0),sum(c.isdigit() for c in url)/max(len(url),1),
        sum(c.isalpha() for c in url)/max(len(url),1),1.0 if suffix in HIGH_RISK_TLDS else 0.0,
        min(subdomain.count('.')+1 if subdomain else 0,5)/5,min(len(domain)/30,1.0),
        1.0 if re.match(r'^(?:\d{1,3}\.){3}\d{1,3}$',ip_host) else 0.0,
        min(sum(b in domain for b in BRAND_KW),3)/3,min(len(path)/200,1.0),
        min(len([s for s in path.split('/') if s])/10,1.0),1.0 if '?' in url else 0.0,
        min(len(query)/200,1.0),1.0 if url.startswith('https') else 0.0,1.0 if 'https' in path else 0.0,
        _ent(url)/6.0,_ent(domain)/4.0,min(sum(kw in url for kw in FINANCIAL_KW),5)/5,
        1.0 if re.search(r'@|//.*@',url) else 0.0,min(url.count('-')/8,1.0),
        1.0 if len(url)>75 and not url.startswith('https') else 0.0,1.0 if suffix in FREE_HOST_TLDS else 0.0,
        min(len(re.findall(r'\d{3,}',url))/3,1.0),
        (1.0 if url.startswith('https') else 0.0)*(0.0 if suffix in HIGH_RISK_TLDS else 1.0)]
print('Feature engineering ready.')

## Cell 3 — Fetch today's phishing URLs from public feeds

Two free, keyless sources:
- **OpenPhish** community feed (text list of phishing URLs)
- **URLhaus** (malware/phishing URL feed from abuse.ch)
Both refresh regularly, so daily runs bring new URLs.

In [ ]:
def fetch_openphish():
    try:
        r=requests.get('https://openphish.com/feed.txt', timeout=30)
        if r.status_code==200:
            return [u.strip() for u in r.text.splitlines() if u.strip().startswith('http')]
    except Exception as e:
        print('  OpenPhish error:', e)
    return []

def fetch_urlhaus():
    try:
        # URLhaus plain-text URL feed (online URLs)
        r=requests.get('https://urlhaus.abuse.ch/downloads/text_online/', timeout=30)
        if r.status_code==200:
            return [u.strip() for u in r.text.splitlines() if u.strip().startswith('http')]
    except Exception as e:
        print('  URLhaus error:', e)
    return []

print('Fetching feeds...')
op=fetch_openphish(); uh=fetch_urlhaus()
print(f'  OpenPhish: {len(op)} URLs')
print(f'  URLhaus:   {len(uh)} URLs')
all_urls=list(dict.fromkeys(op+uh))   # dedup within today
print(f'  Combined unique today: {len(all_urls)}')

## Cell 4 — Filter for MFS-brand URLs + score them

We keep two sets:
1. ALL phishing URLs scored (for the general borderline pool)
2. MFS-brand URLs specifically (the gold set for the rescue result)

In [ ]:
if not all_urls:
    print('No URLs fetched today (feed may be temporarily down). Try again later.')
else:
    # MFS-brand subset (the valuable ones)
    mfs_today=[u for u in all_urls if is_mfs_brand(u)]
    print(f'MFS-brand URLs today: {len(mfs_today)}')

    # score MFS URLs
    rows=[]
    for u in mfs_today:
        try:
            f=np.array([engineer_url_features(u)])
            pp=float(url_model.predict_proba(url_scaler.transform(f))[0,1])
            rows.append({'url':u,'pp':round(pp,4),'date':datetime.now().strftime('%Y-%m-%d')})
        except Exception:
            pass
    mfs_df=pd.DataFrame(rows)
    print(f'Scored {len(mfs_df)} MFS-brand URLs.')
    if len(mfs_df):
        print('  pp range: %.3f - %.3f' % (mfs_df.pp.min(), mfs_df.pp.max()))
        # how many would the URL model MISS (pp<0.5)? those are the rescue candidates
        print(f'  URL model would MISS (pp<0.5): {(mfs_df.pp<0.5).sum()} -> brand detector rescues these')

    # also keep a borderline general pool (pp 0.30-0.70)
    brows=[]
    sample=all_urls[:2000]   # cap to keep runtime sane
    for u in sample:
        try:
            f=np.array([engineer_url_features(u)])
            pp=float(url_model.predict_proba(url_scaler.transform(f))[0,1])
            if 0.30<=pp<=0.70:
                brows.append({'url':u,'pp':round(pp,4),'date':datetime.now().strftime('%Y-%m-%d')})
        except Exception:
            pass
    border_df=pd.DataFrame(brows)
    print(f'Borderline URLs (pp 0.30-0.70) today: {len(border_df)}')

## Cell 5 — Append to master files (dedup, never lose old data)

In [ ]:
def append_dedup(new_df, path, key='url'):
    if new_df is None or len(new_df)==0:
        print(f'  nothing new for {os.path.basename(path)}'); return 0
    if os.path.exists(path):
        old=pd.read_csv(path)
        combined=pd.concat([old,new_df],ignore_index=True).drop_duplicates(subset=[key],keep='first')
        added=len(combined)-len(old)
    else:
        combined=new_df.drop_duplicates(subset=[key]); added=len(combined)
    combined.to_csv(path,index=False)
    return added

if all_urls:
    a1=append_dedup(mfs_df, MFS_MASTER)
    a2=append_dedup(border_df, MASTER)
    print(f'MFS-brand master: +{a1} new  -> {MFS_MASTER}')
    print(f'Borderline master: +{a2} new -> {MASTER}')

    # show growth
    if os.path.exists(MFS_MASTER):
        tot=len(pd.read_csv(MFS_MASTER))
        print(f'\n  TOTAL MFS-brand URLs collected so far: {tot}')
    if os.path.exists(MASTER):
        totb=len(pd.read_csv(MASTER))
        print(f'  TOTAL borderline URLs collected so far: {totb}')
    print('\n  Run again tomorrow to grow the dataset further.')

## What you have + the routine

**Two growing files in v2/data/:**
- `mfs_brand_urls.csv` — real MFS-brand phishing (the gold set for Paper 1's
  rescue result). The ones with pp<0.5 are exactly the cases your brand
  detector rescues.
- `borderline_master.csv` — general borderline URLs (pp 0.30-0.70) for the
  fusion evaluation.

**The daily routine:** open this notebook -> Run all -> close. Once a day.
Each run adds NEW phishing (dedup keeps it clean). After 2-4 weeks you'll
have a much bigger MFS set -> re-run Notebook C for a stronger, tighter
rescue number.

**No need for multiple accounts** — the feeds are public and identical for
everyone; daily runs (not more accounts) is what grows the data.